Install pandas, urllib

In [18]:
#imports
import requests
import pandas as pd
from urllib import parse
import json 
import os
import sys
import datetime 
import logging
from configparser import ConfigParser
from dataclasses import dataclass

In [19]:
#init vars
#this environment var will need to be setup on your system
API_KEY = os.environ['SAP_API_KEY']
print(API_KEY)
#SAP_USER = 'IF_API_TEA' #  IF_API_TEA/IF_API_TEH = TEA, IF_API_TEE = all TEE countries
FORMAT = 'json'
APP_ID = 'f507a79b'
api_url_part1 = 'http://stg-cdsv-odata-3scale-apicast-production.apps.west.cl.ocp.tel.com/v1/odata/'
api_url_part2 = '/sap/opu/odata/sap/'
config_location = "C:\Source\Repos\EnterpriseCloud\Configuration\SAP_cdsv.json"
history_location = "C:\LocalSource\SAP API\CDS_View.config"
data_file_location = 'F:\Files\SAPH-API\\'
api_call_datetimestamp = datetime.datetime.now()
current_timestamp = str(datetime.datetime.now()).replace(" ","_").replace(":","_").replace(".","_")
#if there are a large number of records from the API, write to file instead of memory
writeTempFile = False
tempRecordLimit = 1000

cds_params = {'CDS_VIEW':'',
                  'ACTIVE':'',
                  'CLIENT':'',
                  'SAP_USER':'',
                  'BASE_TABLE':'',
                  'STRATEGY':'',
                  'INCREMENTAL_FIELD':'',
                  'INCREMENTAL_FIELD_TYPE':'',
                  'ROW_LIMIT':'',
                  'SELECT':'',
                  'FILTER':'',
                  'SKIP':'',
                  'TOP':'',
                  'INLINE_COUNT':'',
                  'ORDER_BY':''}

@dataclass
class apiResults:
    row_count: int
    data: dict
    Error_status: int


564a71bc3a18bd04a18f5a7cf33c293b


In [20]:
#user defined vars

#**************** Set these as required ********************

cds_view_name = 'ZCDV_T_CMTU012' #set name of CDS view here 


In [ ]:
#optional
#load view csv file and extract parameters 

In [ ]:
#optional create config
#must have mandatory information as per wiki <link>

In [21]:
#helper functions
#call api
#write file
#write config file

def call_api(client, cds_view, payload):

    #cds view ref needed to passed in url
    cds_ref = cds_view +'_CDS/'
    #build url with cds view
    URL = api_url_part1 + client + api_url_part2 + cds_ref + cds_view

    try:
        response = requests.get(URL, params=payload,timeout=120, stream=True)
        data = None
        status_code = response.status_code
        print(response.url)
        if status_code == 200:
            print("{}: Successful API call".format(status_code))
            results = response.json()
            cds_view_count = results['d']['__count']
            data = results['d']['results']
            #remove __metadata group
            for records in data:
                del records['__metadata']
        elif status_code == 404:
            print("Authentication issue with API call")
            # auth issue, so can't continue, set status to 2         
        elif status_code == 403:
            print("Issue with requested CDS view, check if the cds view is available in this client and transported successfully, Status Code:",status_code)             
            # CDS availability issue, so can't continue, set status to 2
        elif status_code == 502 or status_code == 504:
            print("Timeout or bad response from gateway, try again later, Status Code:",status_code)   
            # timeout issue maybe so set to 1 to reduce page size     
        else:
            print("API Error status code: {}".format(status_code))
            print("API url {}".format(response.url))
            print(message="API Error status code: {}".format(status_code))
        apidata = apiResults(cds_view_count, data,0)
        print("url response time",response.elapsed)
        return apidata
       
    except requests.exceptions.ChunkedEncodingError:
        print('Server did not respond in a timely manor')
        return apiResults(0,None,1)
    except:
        print('Error: ', sys.exc_info()[0])
        return apiResults(0,None,2)


def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def writeConfigParser(path,parserFile):
    f = open(path, 'w') 
    parserFile.write(f)

def writeJsonFile(path,fileOut):
    f = open(path,'w')
    f.write(json.dumps(fileOut))  


In [22]:
#load config into parser and get relevant config for cds view
parser = ConfigParser()
parser.read(config_location)

if parser.has_section(cds_view_name):
        CDS_VIEW  = cds_view_name
        cds_params.update({'CDS_VIEW':CDS_VIEW})
        ACTIVE = parser.getboolean(CDS_VIEW,'active')
        cds_params.update({'ACTIVE':ACTIVE})
        if ACTIVE:
            #check mandatory params are populated
            if parser.get(CDS_VIEW,'sap_user') == '':
                print(message="Sap user not in config file, cannot process cds view {}".format(cds_view_name))
            else:
                cds_params.update({'SAP_USER':parser.get(CDS_VIEW,'sap_user')})

            if parser.get(CDS_VIEW, 'client') == '':
                print(message="API Client not in config file, cannot process cds view {}".format(cds_view_name))
            else:
                cds_params.update({'CLIENT':parser.get(CDS_VIEW, 'client')})

            #operational parameters
            #default row limit to 100000, or restrict to 100000
            if parser.get(CDS_VIEW, 'row_limit') == '' or int(parser.get(CDS_VIEW, 'row_limit'))>100000:
                cds_params.update({'ROW_LIMIT': 100000})
            
            cds_params.update({'STRATEGY':parser.get(CDS_VIEW, 'strategy')}) 
            cds_params.update({'INCREMENTAL_FIELD':parser.get(CDS_VIEW, 'incremental_field')})
            cds_params.update({'BASE_TABLE':parser.get(CDS_VIEW, 'base_table_name')})

            #CDS selection parameters, these are optional
            cds_params.update({'SELECT':parser.get(CDS_VIEW,'select')})
            cds_params.update({'FILTER':parser.get(CDS_VIEW,'filter')})
            cds_params.update({'SKIP':parser.get(CDS_VIEW,'skip')})
            cds_params.update({'TOP':parser.get(CDS_VIEW,'top')})
            #default inlinecount to all pages
            cds_params.update({'INLINE_COUNT':parser.get(CDS_VIEW, 'inlinecount') if parser.get(CDS_VIEW, 'inlinecount') =='' else 'allpages'})
            cds_params.update({'ORDER_BY':parser.get(CDS_VIEW,'orderby')})

            print(cds_params)
        else:
            cds_params = None
            print(message="CDS view found, but not active {}".format(cds_view_name))
else:
    cds_params = None
    print(message="CDS view not found in config file{}".format(cds_view_name))

{'CDS_VIEW': 'ZCDV_T_CMTU012', 'ACTIVE': True, 'CLIENT': 'client224', 'SAP_USER': 'IF_API_SNOWF', 'BASE_TABLE': 'LIKP', 'STRATEGY': 'incremental', 'INCREMENTAL_FIELD': 'Aedat', 'INCREMENTAL_FIELD_TYPE': '', 'ROW_LIMIT': 100000, 'SELECT': '', 'FILTER': '', 'SKIP': '', 'TOP': '', 'INLINE_COUNT': '', 'ORDER_BY': ''}


In [23]:
#create api parameters

#parameters to pass to the API to filter the data retreived from SAP
#default inlinecount to all pages
cds_api_params={'$filter':cds_params.get('FILTER'),
            '$inlinecount':'allpages',
            '$orderby':cds_params.get('ORDER_BY'),
            '$select':cds_params.get('SELECT'),
            '$skip':cds_params.get('SKIP'),
            '$top':cds_params.get('TOP')}

#create payload
#parameters for connecting to the API
payload ={'app_id':APP_ID,'app_key':API_KEY,'sap_user':cds_params.get('SAP_USER'),"$format":FORMAT}  

print("cds_api_params:",cds_api_params)
print("payload:", payload)

cds_api_params: {'$filter': '', '$inlinecount': 'allpages', '$orderby': '', '$select': '', '$skip': '', '$top': ''}
payload: {'app_id': 'f507a79b', 'app_key': '564a71bc3a18bd04a18f5a7cf33c293b', 'sap_user': 'IF_API_SNOWF', '$format': 'json'}


In [ ]:
#****optional****
#Load last run time from cdsv_config and add to filter

history_data = getFile(history_location)

js_data = json.loads(history_data)

if cds_params.get('STRATEGY') == 'incremental':
        history = js_data.get(cds_view_name)
        if history != None:
            payload.update({'$filter':cds_params.get('INCREMENTAL_FIELD')+" ge '"+history+"'"})
            print(cds_view_name,'last successful run',history,'retrieving new data')
        else:
            print('No previous run of CDS view, will attempt to retrieve all data')

In [24]:
#get row count
#init to zero
row_count = 0

#get client from dict
#override client here if required
client = cds_params.get('CLIENT') 
print(client)

#add cds parameters to payload
payload.update({'$inlinecount':'allpages','$top':0})
#requests encodes $, need to prevent that
payload_str = parse.urlencode(payload,safe=':$', encoding='utf-8',quote_via=parse.quote)    
print(payload)    
print("Calling API for row count...")
api_results = call_api(client, cds_view_name, payload_str)
#get row count from API
row_count =  int(api_results.row_count)
print("Records available in cds view:",row_count,"from client:",client)

client224
{'app_id': 'f507a79b', 'app_key': '564a71bc3a18bd04a18f5a7cf33c293b', 'sap_user': 'IF_API_SNOWF', '$format': 'json', '$inlinecount': 'allpages', '$top': 0}
Calling API for row count...
http://stg-cdsv-odata-3scale-apicast-production.apps.west.cl.ocp.tel.com/v1/odata/client224/sap/opu/odata/sap/ZCDV_T_CMTU012_CDS/ZCDV_T_CMTU012?app_id=f507a79b&app_key=564a71bc3a18bd04a18f5a7cf33c293b&sap_user=IF_API_SNOWF&$format=json&$inlinecount=allpages&$top=0
200: Successful API call
url response time 0:00:00.677992
Records available in cds view: 1637 from client: client224


In [15]:
#call api and load data into memory
#add cds parameters to payload
payload.update(cds_api_params)

#get client from dict
#override client here if required
client = cds_params.get('CLIENT') 

#row count controls the number of records that will requested from the API
#default row count to 10 to speed up creation of sample output files
#increase this if required
#if you require all data for a cds view, comment out this row and the row_count from the previous cell will be used
#which is equal to the total number of records in the CDS view
#WARNING - only increase this if absolutely needed, a large number of records will take a considerable amount of time
row_count = 200

#default pagination step, do not change
pagination_step = 50

#default results to empty
api_results= []


#if row count> row_limit then paginate data aquisition
if row_count>pagination_step:
    rows_processed = 0

    # data_file_location = 'C:\\LocalSource\\SAP API\\test2\\'
    file_location = data_file_location + cds_view_name + '.{}.json'.format(current_timestamp)
    tempfile = open(file_location,'w')
        
    while rows_processed < row_count:
        #add a value to the CDS view skip parameter and pagination_step
        payload.update({'$skip':rows_processed,'$top':pagination_step })
        
        #requests encodes $, need to prevent that
        payload_str = parse.urlencode(payload,safe=':$', encoding='utf-8',quote_via=parse.quote)    
        
        print("Calling API for",cds_view_name,"Requesting Rows",rows_processed,"to", rows_processed+pagination_step)
        api_response = call_api(client, cds_view_name, payload_str)
        
        #add data to api_results
        #if no error
        if api_response.Error_status == 0:
            #api_results.extend(api_response.data)
            #write to file after each read
            try:
                tempfile.write(json.dumps(api_response.data))                
                print('wrote to file', rows_processed)
                #api_results=[]
            except:
                print('Error: ', sys.exc_info()[0])

            rows_processed += pagination_step
            print("Api Call Count", int(rows_processed/pagination_step),"Records Processed",rows_processed)
        #api call failed due to server disconnection, reduce pagination step and try again
        elif api_response.Error_status == 1:
            print("API call failed for due to server reset", cds_view_name)
            print('reducing Pagination size from',pagination_step,'to',round(pagination_step/2,-3))
            pagination_step = int(round(pagination_step/2,-3))
            if pagination_step <=1000: break
        #api call failed for unknown reason
        else:
            print("API call failed for unknown reason", cds_view_name)
            #api_results=[]
            break
    tempfile.close()
    

#else call the api normally for all records
else:
    print("Calling API for",cds_view_name,"Requesting all data as row count",(row_count),"< page size")
    payload.update({'$top':row_count})  
    payload_str = parse.urlencode(payload,safe=':$', encoding='utf-8',quote_via=parse.quote)
    
    api_response = call_api(client, cds_view_name, payload_str)
    #check data received ok
    if api_response.Error_status == 0:
        print('Data received ok')
        api_results = api_response.data
    else:
        api_results = None
        print('Error getting data from API')


print("Finished getting Data..")


Calling API for ZCDV_T_CMTU017 Requesting Rows 0 to 50
http://stg-cdsv-odata-3scale-apicast-production.apps.west.cl.ocp.tel.com/v1/odata/client224/sap/opu/odata/sap/ZCDV_T_CMTU017_CDS/ZCDV_T_CMTU017?app_id=f507a79b&app_key=564a71bc3a18bd04a18f5a7cf33c293b&sap_user=IF_API_SNOWF&$format=json&$inlinecount=allpages&$top=50&$filter=&$orderby=&$select=&$skip=0
200: Successful API call
url response time 0:00:00.779109
wrote to file 0
Api Call Count 1 Records Processed 50
Calling API for ZCDV_T_CMTU017 Requesting Rows 50 to 100
http://stg-cdsv-odata-3scale-apicast-production.apps.west.cl.ocp.tel.com/v1/odata/client224/sap/opu/odata/sap/ZCDV_T_CMTU017_CDS/ZCDV_T_CMTU017?app_id=f507a79b&app_key=564a71bc3a18bd04a18f5a7cf33c293b&sap_user=IF_API_SNOWF&$format=json&$inlinecount=allpages&$top=50&$filter=&$orderby=&$select=&$skip=50
200: Successful API call
url response time 0:00:00.758723
wrote to file 50
Api Call Count 2 Records Processed 100
Calling API for ZCDV_T_CMTU017 Requesting Rows 100 to 150

In [17]:
#write output of api to file
    
if api_results != []:
    print("writing cds view",cds_view_name," data to file")
    file_location = data_file_location + cds_view_name + '.{}.json'.format(current_timestamp)
    writeJsonFile(file_location,api_results)
    print("data written to file:", file_location)
    #clear
    api_results = None
else:
    print("no date returned from API for cds view",cds_view_name)



writing cds view ZCDV_T_CMTU017  data to file
data written to file: F:\Files\SAPH-API\ZCDV_T_CMTU017.2022-08-30_07_48_50_051945.json


In [ ]:
#optional
#if last run for incremental load, update history

field_type = client = cds_params.get('INCREMENTAL_FIELD') 

#create a timestamp depending on the type in the cds view config
if field_type == 'date':
    current_datetimestamp = datetime.datetime.strftime(api_call_datetimestamp,'%Y%m%d')
elif field_type == 'datetime':
    current_datetimestamp = datetime.datetime.strftime(api_call_datetimestamp,'%Y%m%d%H%M%S')
else:
    current_datetimestamp = datetime.datetime.strftime(api_call_datetimestamp,'%Y%m%d%H%M%S')

#init to empty
js_data ={}

#if the file exists, open it and update/create last run
#else create file and add tuple
if os.path.getsize(file_location) > 0:
    with open(file_location) as f:
        data = f.read()

        js_data = json.loads(data)
        js_data.update({cds_view_name:current_datetimestamp})
else:
    js_data = {cds_view_name:current_datetimestamp}

#write to file
writeJsonFile(file_location,js_data)
